In [1]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="/kaggle/input/datasets/priyansh384/nonconv/nonconv_hindi_medical_20M_clean.jsonl")

# Keep only text
def preprocess(example):
    return {"text": example["text"]}

dataset = dataset.map(preprocess)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/8345 [00:00<?, ? examples/s]

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
from huggingface_hub import login
login("")

In [4]:
pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/gemma-2b"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)

model.config.use_cache = False

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [6]:
def tokenize(batch):
    out = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    out["labels"] = out["input_ids"].copy()
    return out

tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/8345 [00:00<?, ? examples/s]

In [7]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj","k_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,843,200 || all params: 2,508,015,616 || trainable%: 0.0735


In [8]:
import torch
model = model.to("cuda:0")
torch.cuda.set_device(0)

In [9]:
from transformers.utils import logging
logging.set_verbosity_info()

In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/kaggle/working/gemma-hindi-med",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,

    logging_strategy="steps",     # show running loss
    logging_steps=10,             # print every 10 steps

    save_strategy="epoch",        # save model each epoch
         # change later if eval set

    fp16=True,
    report_to="none",
    remove_unused_columns=True
)

PyTorch: setting up devices


In [11]:
from transformers import Trainer, default_data_collator

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=default_data_collator
)

In [12]:
trainer.train()

***** Running training *****
  Num examples = 8,345
  Num Epochs = 3
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 8
  Total optimization steps = 3,132
  Number of trainable parameters = 1,843,200


Step,Training Loss
10,4.923183
20,5.151616
30,4.654482
40,5.826865
50,4.528748
60,5.441961
70,5.428794
80,4.240628
90,6.305247
100,5.925482


Saving model checkpoint to /kaggle/working/gemma-hindi-med/checkpoint-1044
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--google--gemma-2b/snapshots/9cf48e52b224239de00d483ec8eb84fb8d0f3a3a/config.json
Model config GemmaConfig {
  "architectures": [
    "GemmaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 2,
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "head_dim": 256,
  "hidden_act": "gelu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 16384,
  "max_position_embeddings": 8192,
  "model_type": "gemma",
  "num_attention_heads": 8,
  "num_hidden_layers": 18,
  "num_key_value_heads": 1,
  "pad_token_id": 0,
  "rms_norm_eps": 1e-06,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "use_bidirectional_attention": null,
  "use_cache": true,
  "vocab_size": 256000
}

Saving mode

TrainOutput(global_step=3132, training_loss=4.893971079306279, metrics={'train_runtime': 11160.7364, 'train_samples_per_second': 2.243, 'train_steps_per_second': 0.281, 'total_flos': 1.5256357130207232e+17, 'train_loss': 4.893971079306279, 'epoch': 3.0})

In [13]:
model.save_pretrained("/kaggle/working/medgemma-hindi-lora")
tokenizer.save_pretrained("/kaggle/working/medgemma-hindi-lora")

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--google--gemma-2b/snapshots/9cf48e52b224239de00d483ec8eb84fb8d0f3a3a/config.json
Model config GemmaConfig {
  "architectures": [
    "GemmaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 2,
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "head_dim": 256,
  "hidden_act": "gelu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 16384,
  "max_position_embeddings": 8192,
  "model_type": "gemma",
  "num_attention_heads": 8,
  "num_hidden_layers": 18,
  "num_key_value_heads": 1,
  "pad_token_id": 0,
  "rms_norm_eps": 1e-06,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "use_bidirectional_attention": null,
  "use_cache": true,
  "vocab_size": 256000
}

tokenizer config file saved in /kaggle/working/medgemma-hindi-lora/tokenizer_config.js

('/kaggle/working/medgemma-hindi-lora/tokenizer_config.json',
 '/kaggle/working/medgemma-hindi-lora/tokenizer.json')